# XTrendLL — Workflow A2 (learnable τ **+** Bennett static prior)

Same as `xtrendll_a1_no_bennett.ipynb`, but the lag-aware peer block additionally receives a **pre-fitted Bennett lead-lag matrix** `S: [N, N]` as a non-trainable buffer. Each `(peer v, lag τ)` logit gets an additive bias `α · log(S[v, u] + ε)`. `α` is a learnable scalar initialised at `0.1` — the prior is a soft guide.

**Prerequisite.** `prep_artifacts.ipynb` must have been run with `BUILD_BENNETT = True` (that produces `lead_lag_matrix__<tag>.pkl`). The cell that loads artefacts below asserts this.

**Self-contained.** Only the `xtrendll` fork repo is cloned — no base repo needed. Any Colab GPU works (T4 / L4 / A100 / H100).

In [ ]:
# ── GPU check ───────────────────────────────────────────────────────────
import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU not detected!')
print(f'PyTorch {torch.__version__} | GPU: {torch.cuda.get_device_name(0)}')
print(f'CUDA: {torch.version.cuda}   bf16 supported: {torch.cuda.is_bf16_supported()}')

In [ ]:
# ── Clone the xtrendll fork + install deps ──────────────────────────────
import os, sys

XTRENDLL_REPO = 'https://github.com/YOUR-USERNAME/xtrendll.git'   # <-- EDIT ME

%cd /content
!rm -rf xtrendll
!git clone $XTRENDLL_REPO

for var in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS',
            'VECLIB_MAXIMUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[var] = '1'

!pip install -q numpy pandas scipy yfinance tqdm matplotlib
sys.path.insert(0, '/content')
!ls /content/xtrendll/*.py

In [ ]:
# ── Mount Drive + paths ─────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

ARTIFACTS_DIR = '/content/drive/MyDrive/xtrendll_artifacts_v1_etf50'
RESULTS_ROOT  = '/content/drive/MyDrive/results_xtrendll'

assert os.path.isdir(ARTIFACTS_DIR), (
    f'Artefacts not found at {ARTIFACTS_DIR}. '
    'Run prep_artifacts.ipynb locally with BUILD_BENNETT=True and upload the folder.'
)
print('Artefacts:', ARTIFACTS_DIR)

In [ ]:
# ── Imports (all from xtrendll) ─────────────────────────────────────────
from copy import deepcopy
from pathlib import Path
import random, time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from xtrendll.config import DATA, MODEL, TRAIN, CPD
from xtrendll.data import build_episode_loaders, time_split
from xtrendll.train import fit, eval_epoch
from xtrendll.backtest import (
    run_backtest, compare_equity, print_comparison, build_benchmarks,
)

from xtrendll import (
    XTrendLL, LL_DEFAULT, _xtrendll_step, load_artifacts, save_run,
)
from xtrendll.lead_lag import align_S_to_asset_ids

print('imports OK')

In [ ]:
# ── Seeds + run config ──────────────────────────────────────────────────
SEED = DATA['seed']
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED); random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

DATA_RUN  = deepcopy(DATA)
TRAIN_RUN = deepcopy(TRAIN)
MODEL_RUN = deepcopy(MODEL)
LL_RUN    = deepcopy(LL_DEFAULT)

# A2 workflow: Bennett ON.
LL_RUN['use_bennett'] = True
LL_RUN['alpha_init']  = 0.1

MODEL_RUN['hidden_dim']  = 128
DATA_RUN['num_context']  = 10
MAX_PEERS                = 10

print('LL_RUN:', LL_RUN)

In [ ]:
# ── Load artefacts (must include Bennett S) ────────────────────────────
art = load_artifacts(ARTIFACTS_DIR)
if art['lead_lag_matrix'] is None:
    raise FileNotFoundError(
        'No lead_lag_matrix in the artefacts bundle. '
        'Re-run prep_artifacts.ipynb with BUILD_BENNETT=True and re-upload.'
    )

panel          = art['panel']
fcols          = art['fcols']
tk2id          = art['tk2id']
train_regimes  = art['train_regimes']
val_cache      = art['val_regime_cache']
test_cache     = art['test_regime_cache']
ll_payload     = art['lead_lag_matrix']

DATA_RUN['tickers'] = list(tk2id.keys())
DATA_RUN['start']   = art['data_run']['start']
DATA_RUN['end']     = art['data_run']['end']

n_assets = len(tk2id)
n_feat   = len(fcols)

# Re-order S so rows/cols index by asset_id, matching the model embedding.
S_aligned = align_S_to_asset_ids(ll_payload, tk2id, num_assets=n_assets)
print(f'S aligned to asset_id space: shape={S_aligned.shape}')
print(f'  density={(S_aligned > 0).mean():.2%}  '
      f'mean(positive)={S_aligned[S_aligned > 0].mean():.3f}  '
      f'max={S_aligned.max():.3f}')

train_d, val_d, test_d = time_split(panel, DATA_RUN['train_frac'], DATA_RUN['val_frac'])
print(f'n_assets={n_assets}  train={len(train_d)}  val={len(val_d)}  test={len(test_d)}')

In [ ]:
# ── Build loaders (peers ON) ───────────────────────────────────────────
sets, loaders = build_episode_loaders(
    panel, fcols, train_d, val_d, test_d,
    train_regimes, DATA_RUN,
    regime_caches={'val': val_cache, 'test': test_cache},
    include_peers=True,
    max_peers=MAX_PEERS,
)
print(f'batches  train={len(loaders["train"])}  val={len(loaders["val"])}  test={len(loaders["test"])}')

In [ ]:
# ── Instantiate XTrendLL with Bennett S matrix ─────────────────────────
device = 'cuda'
S_tensor = torch.from_numpy(S_aligned).float()

model = XTrendLL(
    input_dim=n_feat, num_assets=n_assets,
    cfg=MODEL_RUN, ll_cfg=LL_RUN, S_matrix=S_tensor,
).to(device)

n_params_total = sum(p.numel() for p in model.parameters())
n_params_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params:     {n_params_total:,}')
print(f'Trainable params: {n_params_train:,}')
print('alpha (Bennett bias scale):',
      float(model.cs_block.alpha) if model.cs_block.alpha is not None else 'None')

In [ ]:
# ── Forward sanity check ───────────────────────────────────────────────
batch = next(iter(loaders['train']))
model.eval()
with torch.no_grad():
    b = {k: (v.to(device) if isinstance(v, torch.Tensor) else v) for k, v in batch.items()}
    pos = model(
        b['target_x'], b['target_id'],
        b['ctx_x'],   b['ctx_y'],    b['ctx_id'],
        b['peer_x'],  b['peer_id'],  b['peer_mask'],
    )
    assert pos.shape == (b['target_x'].shape[0], DATA_RUN['lookback'])
print('forward OK, pos.shape =', tuple(pos.shape))

In [ ]:
# ── Train ──────────────────────────────────────────────────────────────
model.train()
t_start = time.perf_counter()
model, history = fit(
    model, loaders['train'], loaders['val'], device,
    _xtrendll_step, TRAIN_RUN, MODEL_RUN,
)
elapsed = time.perf_counter() - t_start
print(f'training elapsed: {elapsed:.1f}s  (α final = {float(model.cs_block.alpha):.3f})')
history.tail(10)

In [ ]:
# ── Evaluate + backtest ────────────────────────────────────────────────
test_results = eval_epoch(
    model, loaders['test'], device, MODEL_RUN['warmup_steps'], _xtrendll_step,
    cost_bps=TRAIN_RUN['cost_bps'],
)
pred_df = test_results['pred_df']
backtest   = run_backtest(pred_df, cost_bps=TRAIN_RUN['cost_bps'], label='XTrendLL A2')
benchmarks = build_benchmarks(pred_df)
equity_fig = compare_equity([backtest], benchmarks, title='XTrendLL A2 vs benchmarks')
plt.show()
comparison_df = print_comparison([backtest, *benchmarks])

In [ ]:
# ── Persist the run ────────────────────────────────────────────────────
cfg_snapshot = {
    'DATA':   DATA_RUN,
    'MODEL':  MODEL_RUN,
    'TRAIN':  TRAIN_RUN,
    'CPD':    dict(CPD),
    'LL':     LL_RUN,
    'MAX_PEERS': MAX_PEERS,
    'artifacts_tag': art.get('tag'),
    'bennett_max_lag': ll_payload.get('max_lag'),
    'S_aligned_shape': list(S_aligned.shape),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
}

run_dir = save_run(
    out_root     = RESULTS_ROOT,
    run_tag      = f'xtrendll_a2__{art.get("tag", "v1")}',
    model        = model,
    history      = history,
    test_results = test_results,
    backtest     = backtest,
    cfg_snapshot = cfg_snapshot,
    benchmarks   = benchmarks,
    comparison_df= comparison_df,
    equity_fig   = equity_fig,
    seed         = SEED,
    extras       = {
        'elapsed_seconds': elapsed,
        'n_params': n_params_train,
        'alpha_final': float(model.cs_block.alpha) if model.cs_block.alpha is not None else None,
    },
)
print('Run saved to', run_dir)